In [1]:
import torch
import torch.nn as nn
class ShortcutScopeCNN(nn.Module):
    def __init__(self, num_classes = 2):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(), 
            nn.MaxPool2d(kernel_size=2)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(), 
            nn.MaxPool2d(kernel_size=2)
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size = 3, padding = 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=0.3),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x
if __name__ == "__main__":
    model = ShortcutScopeCNN(num_classes=2)
    dummy_images = torch.randn(32, 3, 128, 128)
    outputs = model(dummy_images)

    print("Output shape:", outputs.shape)
    print("Total parameters:", sum(
        p.numel() for p in model.parameters()
    ))
    print("CNN architecture works!")

Output shape: torch.Size([32, 2])
Total parameters: 389890
CNN architecture works!


In [2]:
from torchinfo import summary

# Instantiate your model
model = ShortcutScopeCNN(num_classes=2)

# Pass the input tensor shape: (batch_size, channels, height, width)
summary(
    model, 
    input_size=(1, 3, 128, 128),
    col_names=["input_size", "output_size", "num_params", "trainable"],
    col_width=20,
    row_settings=["var_names"]
)


Layer (type (var_name))                  Input Shape          Output Shape         Param #              Trainable
ShortcutScopeCNN (ShortcutScopeCNN)      [1, 3, 128, 128]     [1, 2]               --                   True
├─Sequential (block1)                    [1, 3, 128, 128]     [1, 32, 64, 64]      --                   True
│    └─Conv2d (0)                        [1, 3, 128, 128]     [1, 32, 128, 128]    896                  True
│    └─BatchNorm2d (1)                   [1, 32, 128, 128]    [1, 32, 128, 128]    64                   True
│    └─ReLU (2)                          [1, 32, 128, 128]    [1, 32, 128, 128]    --                   --
│    └─MaxPool2d (3)                     [1, 32, 128, 128]    [1, 32, 64, 64]      --                   --
├─Sequential (block2)                    [1, 32, 64, 64]      [1, 64, 32, 32]      --                   True
│    └─Conv2d (0)                        [1, 32, 64, 64]      [1, 64, 64, 64]      18,496               True
│    └─BatchNorm2d